# Step 1: Environment Setup

In [ ]:
!pip install langchain langchain-openai langchain-community gradio langchain_google_genai

# Step 2: Configuration

### Set up Google API Key



In [ ]:
import os
import json
import gradio as gr

import google.generativeai as genai
from google.colab import userdata

# The previous try-except block for 'google.genai' was causing an AttributeError
# for list_models. We will consistently use 'google.generativeai' as 'genai'.

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Configure the genai library with the API key
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the LLM with Google Gemini
# You will need to set your GOOGLE_API_KEY in Colab secrets
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7, google_api_key=GOOGLE_API_KEY)

# Step 3&4: Design Prompts & Function Schema

In [ ]:
# Function Schema
tools = [{
    "type": "function",
    "function": {
        "name": "extract_candidate_info",
        "description": "Extract candidate information for job applications",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "desired_role": {"type": "string"},
                "experience_years": {"type": "integer"}
            },
            "required": ["name", "desired_role"]
        }
    }
}]

# System Prompt
system_prompt = """
You are a friendly recruitment assistant. You help job seekers with career questions and extract their info.

Examples:
User: How do I improve my resume?
Assistant: Focus on quantifiable achievements! Use bullet points to highlight your impact.

User: I am Sarah, applying for a Data Scientist role with 5 years of experience.
Assistant: {{"name": "Sarah", "desired_role": "Data Scientist", "experience_years": 5}}

Always return structured JSON ONLY when the user provides personal application details.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Step 5&6: Build the Chatbot

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 1. Setup Memory Storage
# We use a dictionary to hold history sessions for the app
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 2. Define the Chain
# We bind tools to the LLM and create the full chain
llm_with_tools = llm.bind_tools(tools)
chain = prompt | llm_with_tools

# 3. Create the Runnable with Message History
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 4. Define the final interface function
def chat_function(message, history):
    # Execute the chain
    response = chain_with_history.invoke(
        {"input": message},
        config={"configurable": {"session_id": "recruitment_session"}}
    )

    # Check if a tool was called (Extraction case)
    if response.tool_calls:
        # Extract and return the JSON
        return json.dumps(response.tool_calls[0]['args'])

    # Otherwise return normal conversational content
    return response.content

# Launch Gradio
demo = gr.ChatInterface(chat_function, title="Recruitment Assistant")
demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://306816e5701839a615.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://306816e5701839a615.gradio.live


# Step 7: Test Cases

Chatbot testing:

Conversational 1: "How do I prepare for a data analyst interview?"

Conversational 2: "What is the job market like for designers?"

Conversational 3: "What should I include in a cover letter?"

Extraction 1: "Hi, I'm Alex. I want to be a Project Manager and I have 4 years of experience."

Extraction 2: "My name is Jordan, applying for a Backend Developer role, 2 years experience."

In [ ]:
# Function to simulate the testing process
def run_test_cases(chatbot_func):
    test_inputs = [
        "How do I prepare for a data analyst interview?",
        "What is the job market like for designers?",
        "What should I include in a cover letter?",
        "Hi, I'm Alex. I want to be a Project Manager and I have 4 years of experience.",
        "My name is Jordan, applying for a Backend Developer role, 2 years experience."
    ]

    results = {}
    for i, user_input in enumerate(test_inputs, 1):
        # We pass None as history for simplicity in this test loop
        response = chatbot_func(user_input, [])
        results[f"Test Case {i}"] = {
            "Input": user_input,
            "Output": response
        }

    return results

# Execute tests
test_results = run_test_cases(chat_function)

# Display results
for test_name, data in test_results.items():
    print(f"--- {test_name} ---")
    print(f"Input: {data['Input']}")
    print(f"Output: {data['Output']}\n")

--- Test Case 1 ---
Input: How do I prepare for a data analyst interview?
Output: [{'type': 'text', 'text': "To prepare for a data analyst interview, focus on these key areas:\n\n1.  **SQL Skills:** Be ready to write and debug SQL queries. Practice common operations like joins, aggregations, and subqueries.\n2.  **Excel/Spreadsheet Skills:** Understand how to manipulate data, use formulas, and create pivot tables.\n3.  **Statistical Concepts:** Brush up on descriptive statistics, hypothesis testing, and A/B testing.\n4.  **Data Visualization:** Be familiar with tools like Tableau or Power BI and be able to discuss best practices for creating effective visualizations.\n5.  **Case Studies:** Practice problem-solving data-related scenarios. Think about how you would approach a business problem using data.\n6.  **Behavioral Questions:** Prepare to discuss your experience, how you handle challenges, and why you're interested in the role.\n\nGood luck!", 'extras': {'signature': 'CrYDAQw51seA

# Step 8: Reflection

Reflection on Advanced Prompt Engineering

Function calling significantly improved response consistency by enforcing a strict schema for sensitive data. Without it, the LLM might hallucinate conversational filler around the JSON, breaking downstream integration. By defining extract_candidate_info as a tool, I moved the responsibility of data parsing from the LLM’s text generation layer to a structured execution layer, ensuring valid output every time.

Prompt engineering, specifically the use of few-shot prompting, was crucial in balancing tone and formatting. By providing examples, I established a "friendly" persona while explicitly showing the model when to output natural language versus machine-readable JSON. This separation of concerns prevents the chatbot from becoming too robotic, while still remaining useful for automated HR systems.

For a production version, I would add Schema Validation using Pydantic. Currently, the tool output is a raw dictionary; using a Pydantic model would allow me to enforce constraints (e.g., experience_years cannot be negative) before the data ever reaches the database. Additionally, I would implement Human-in-the-loop verification, where extracted JSON is summarized and displayed to the user for confirmation before being finalized in the HR system.